In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.neighbors import KNeighborsRegressor

RANDOM_STATE = 42

ModuleNotFoundError: No module named 'matplotlib'

# Load Model & Preprocessing

In [ ]:
file_path = 'ae_only_unambiguous_1000.csv'
df = pd.read_csv(file_path, encoding='utf-8', encoding_errors='ignore', dtype={'lang5.x': str, 'lang6.x': str})

In [ ]:
df_target = df.groupby('website')['response.x'].mean().reset_index()

df_target = df_target.rename(columns={
    'website': 'image_name',
    'response.x': 'skor_estetika'
})

df_target['image_name'] = df_target['image_name'] + '.png'
df_target.head(5)

,image_name,skor_estetika
0,english_0.png,2.290287
1,english_1.png,4.195294
2,english_10.png,3.926130
3,english_100.png,4.958242
4,english_101.png,4.864776


In [ ]:
df_target.shape

(418, 2)

In [ ]:
df_cnn = pd.read_csv('ekstrak_cnn.csv')
df_cv = pd.read_csv('ekstrak_matematika_opencv.csv')

df_hybrid = pd.merge(df_cnn, df_cv, on='image_name', how='inner')
df_hybrid.head(5)

,image_name,cnn_feat_0,cnn_feat_1,cnn_feat_2,cnn_feat_3,cnn_feat_4,cnn_feat_5,cnn_feat_6,cnn_feat_7,cnn_feat_8,...,purple,fuchsia,green,lime,olive,yellow,navy,blue,teal,aqua
0,english_0.png,-4.721025,2.358559,-3.684831,0.463756,-0.524554,1.350653,-0.075894,1.700438,0.542055,...,0.000000,0.000000,0.000544,0.000000,0.000547,0.000252,0.019568,0.000340,0.005155,0.001189
1,english_1.png,1.210117,5.665122,-3.581102,-1.811846,-4.932022,0.047140,-0.264386,1.231203,0.168720,...,0.000048,0.000160,0.000078,0.000011,0.002449,0.002518,0.001640,0.000699,0.075212,0.003403
2,english_10.png,-2.391359,3.296005,0.787008,0.606368,0.721851,0.919729,0.678484,-0.327567,-0.328909,...,0.007382,0.000051,0.000003,0.000000,0.001636,0.001023,0.009420,0.000594,0.015256,0.006435
3,english_100.png,-0.749734,0.351490,4.590096,2.475029,-1.191490,-3.183770,3.293322,-1.044738,0.705354,...,0.000027,0.000000,0.000000,0.000000,0.085815,0.054267,0.000006,0.000000,0.005960,0.000017
4,english_101.png,4.423751,-1.595877,-1.300480,1.046409,0.702248,3.977708,-0.080589,-2.913515,1.900459,...,0.000015,0.000000,0.021142,0.000000,0.008461,0.000000,0.153999,0.000011,0.057641,0.000156


In [ ]:
df_hybrid.shape

(410, 53)

In [ ]:
df_final = pd.merge(df_hybrid, df_target, on='image_name', how='inner')
df_final.head(5)

,image_name,cnn_feat_0,cnn_feat_1,cnn_feat_2,cnn_feat_3,cnn_feat_4,cnn_feat_5,cnn_feat_6,cnn_feat_7,cnn_feat_8,...,fuchsia,green,lime,olive,yellow,navy,blue,teal,aqua,skor_estetika
0,english_0.png,-4.721025,2.358559,-3.684831,0.463756,-0.524554,1.350653,-0.075894,1.700438,0.542055,...,0.000000,0.000544,0.000000,0.000547,0.000252,0.019568,0.000340,0.005155,0.001189,2.290287
1,english_1.png,1.210117,5.665122,-3.581102,-1.811846,-4.932022,0.047140,-0.264386,1.231203,0.168720,...,0.000160,0.000078,0.000011,0.002449,0.002518,0.001640,0.000699,0.075212,0.003403,4.195294
2,english_10.png,-2.391359,3.296005,0.787008,0.606368,0.721851,0.919729,0.678484,-0.327567,-0.328909,...,0.000051,0.000003,0.000000,0.001636,0.001023,0.009420,0.000594,0.015256,0.006435,3.926130
3,english_100.png,-0.749734,0.351490,4.590096,2.475029,-1.191490,-3.183770,3.293322,-1.044738,0.705354,...,0.000000,0.000000,0.000000,0.085815,0.054267,0.000006,0.000000,0.005960,0.000017,4.958242
4,english_101.png,4.423751,-1.595877,-1.300480,1.046409,0.702248,3.977708,-0.080589,-2.913515,1.900459,...,0.000000,0.021142,0.000000,0.008461,0.000000,0.153999,0.000011,0.057641,0.000156,4.864776


In [ ]:
df_final.shape

(398, 54)

In [ ]:
X_cv = pd.merge(df_cv, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cv.shape

(398, 22)

In [ ]:
X_cnn = pd.merge(df_cnn, df_target, on='image_name', how='inner').drop(columns=['image_name', 'skor_estetika'])
X_cnn.shape

(398, 30)

In [ ]:
X = df_final.drop(columns=['image_name', 'skor_estetika'])
y = df_final['skor_estetika']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Building Model

## Linear Regression

In [ ]:
lr_model = LinearRegression()

t0 = time.perf_counter()
lr_model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = lr_model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"R2 Score        : {r2:.4f}")
print(f"RMSE            : {rmse:.4f}")
print(f"Waktu training  : {waktu_train:.4f} detik")
print(f"Waktu prediksi  : {waktu_predict:.4f} detik")

r2_train = r2_score(y_train, lr_model.predict(X_train_scaled))
print(f"R2 Score (train) : {r2_train:.4f}")


R2 Score        : 0.2715
RMSE            : 0.7861
Waktu training  : 0.0525 detik
Waktu prediksi  : 0.0008 detik
R2 Score (train) : 0.6507


In [ ]:
feature_pipeline = Pipeline([
    ('transformer', PowerTransformer(method='yeo-johnson', standardize=True)),
    ('lr', LinearRegression())
])

model = TransformedTargetRegressor(
    regressor=feature_pipeline,
    transformer=PowerTransformer(method='yeo-johnson', standardize=True)
)

t0 = time.perf_counter()
model.fit(X_train_scaled, y_train)
waktu_train = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred = model.predict(X_test_scaled)
waktu_predict = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2_train = r2_score(y_train, model.predict(X_train_scaled))

print(f"R2 Score          : {r2:.4f}")
print(f"RMSE              : {rmse:.4f}")
print(f"Waktu training    : {waktu_train:.4f} detik")
print(f"Waktu prediksi    : {waktu_predict:.4f} detik")
print(f"R2 Score (train)  : {r2_train:.4f}")

## K-Nearest-Neighbours

In [ ]:
knn = KNeighborsRegressor(n_neighbors=5, weights='uniform')

t0 = time.perf_counter()
knn.fit(X_train_scaled, y_train)
waktu_train_knn = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_knn = knn.predict(X_test_scaled)
waktu_predict_knn = time.perf_counter() - t0

r2 = r2_score(y_test, y_pred_knn)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_knn))

print(f"R2 Score           : {r2:.4f}")
print(f"RMSE               : {rmse:.4f}")
print(f"Waktu training     : {waktu_train_knn:.4f} detik")
print(f"Waktu prediksi     : {waktu_predict_knn:.4f} detik")

r2_train = r2_score(y_train, knn.predict(X_train_scaled))
print(f"R2 Score (train) : {r2_train:.4f}")

R2 Score           : 0.2579
RMSE               : 0.7934
Waktu training     : 0.0022 detik
Waktu prediksi     : 0.1201 detik
R2 Score (train) : 0.4423


# Evaluation

# Model Overfitting

In [ ]:
kolom_cnn = [col for col in df_hybrid.columns if 'cnn_feat' in col]
kolom_cv = [col for col in df_hybrid.columns if col not in kolom_cnn and col not in ['image_name', 'skor_estetika']]

matriks_korelasi = df_hybrid[kolom_cnn + kolom_cv].corr()

for i in range(30):
  target_fitur = f'cnn_feat_{str(i)}'
  korelasi_feat_7 = matriks_korelasi.loc[target_fitur, kolom_cv]
  korelasi_feat_7_sorted = korelasi_feat_7.abs().sort_values(ascending=False)

  print("\n======================================================")
  print(f"Korelasi tertinggi untuk {target_fitur} dengan fitur OpenCV:")
  print(korelasi_feat_7.loc[korelasi_feat_7_sorted.index].head())


Korelasi tertinggi untuk cnn_feat_0 dengan fitur OpenCV:
white              -0.535615
quadtree_leaves     0.517448
image_area_ratio    0.411305
colorfulness        0.335544
gray                0.291772
Name: cnn_feat_0, dtype: float64

Korelasi tertinggi untuk cnn_feat_1 dengan fitur OpenCV:
white               0.482866
black              -0.430684
symmetry           -0.345834
maroon             -0.304175
image_area_ratio    0.217094
Name: cnn_feat_1, dtype: float64

Korelasi tertinggi untuk cnn_feat_2 dengan fitur OpenCV:
black     -0.365272
white      0.340232
teal      -0.233975
balance    0.201654
yellow     0.170520
Name: cnn_feat_2, dtype: float64

Korelasi tertinggi untuk cnn_feat_3 dengan fitur OpenCV:
quadtree_leaves    -0.532910
colorfulness       -0.386566
red                -0.323567
image_area_ratio   -0.225984
maroon             -0.208791
Name: cnn_feat_3, dtype: float64

Korelasi tertinggi untuk cnn_feat_4 dengan fitur OpenCV:
black               0.298145
quadtree_leave

# Result Analysis